# AI 3 · 3주차 정확도 개선·예외 처리·평가

2주차 모듈에 위험 판단 예외 처리와 제품명 개선을 더하고, **판단 보류 기준값별 위험 놓침률·보류율**을 잽니다.
AI 2가 찍은 실제 사진(매운새우깡)의 OCR 결과로 재고, AI 4 안내 문장까지 이어지는지 확인합니다.
팀 사진 10종의 OCR 결과와 정답이 나오면 5번 셀의 경로만 바꿔서 다시 실행하면 됩니다.

**실행 방법:** 메뉴에서 **런타임 → 모두 실행**을 누르면 됩니다. GPU는 필요 없고 **1~2분** 걸립니다.

## 1. 레포 받아오기

GitHub에서 코드를 고쳤다면 이 셀부터 다시 실행해야 바뀐 내용이 반영됩니다.

In [ ]:
REPO_URL = "https://github.com/DL-2026F-Team05/package-info-extraction.git"

import os, subprocess, sys

os.chdir("/content")
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")   # 비공개 레포일 때만 필요
except Exception:
    token = None
url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL

if not os.path.exists("package-info-extraction"):
    if subprocess.run(["git", "clone", "-q", url, "package-info-extraction"]).returncode != 0:
        raise RuntimeError("레포를 받아오지 못했습니다. REPO_URL과 공개/비공개 설정(토큰)을 확인하세요.")
os.chdir("/content/package-info-extraction")
subprocess.run(["git", "pull", "-q"])
sys.path.insert(0, os.getcwd())
print("현재 위치:", os.getcwd())

## 2. 패키지 설치

`pydantic`은 코랩에 이미 들어 있어서 보통 바로 끝납니다.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("설치 완료")

## 3. 테스트

날짜 형식, 알레르기 매칭, 위험 판단, 실제 사진 OCR 결과를 확인합니다. 모두 `passed`가 나와야 합니다.

In [ ]:
!python -m pytest -q

## 4. AI 2 OCR 결과로 정보 추출

AI 2 `run_pipeline` 결과 JSON을 AI 2 문서에 적힌 대로 `extract(result["ocr"], user_allergens)`에 넣습니다.
`USER_ALLERGENS`를 바꿔 보세요.

In [ ]:
import json
from pathlib import Path
from extraction import extract
from extraction.__main__ import summary

USER_ALLERGENS = ["우유", "땅콩"]
TODAY = "2026-11-17"

for f in sorted(Path("data/samples/ai2_week2").glob("*.json")):
    result = json.loads(f.read_text(encoding="utf-8"))
    info = extract(result["ocr"], USER_ALLERGENS, TODAY)
    info["image"] = f.name
    print(summary(info), end="\n\n")

## 5. 평가

정답 라벨과 비교해 항목별 정확도, F1, 위험 오분류율을 계산하고, 판단 보류 기준값을 0~0.95로 바꿔 가며
**위험 놓침률·보류율**을 잽니다. 결과는 `results/ai3_week3/`에 저장됩니다.
팀 사진으로 잴 때는 `OCR_DIR`(OCR 결과 JSON 폴더)과 `LABELS`(제품 단위 정답)만 바꾸세요.

In [ ]:
from IPython.display import Image, Markdown, display
from extraction.evaluate import main as evaluate

OCR_DIR = "data/samples/ai2_week2"
LABELS = "data/labels/labels.csv"
evaluate(["--ocr-dir", OCR_DIR, "--labels", LABELS, "--today", TODAY, "--sweep", "--out", "results/ai3_week3"])
display(Markdown(Path("results/ai3_week3/summary.md").read_text(encoding="utf-8")))
display(Image("results/ai3_week3/sweep.png"))

## 6. AI 4와 연결 (안내 문장까지)

AI 4 레포(AI_tts_web)를 받아 와서, AI 3 결과를 AI 4 안내 문장(`speech.compose`)에 그대로 넣어 봅니다.
마지막 줄은 AI_tts_web 의 테스트를 이 모듈로 돌려 보는 확인입니다 (AI_tts_web 파일은 바꾸지 않고 복사본에서만).

In [ ]:
AI4 = "/content/AI_tts_web"
if not os.path.exists(AI4):
    subprocess.run(["git", "clone", "-q", "https://github.com/DL-2026F-Team05/AI_tts_web.git", AI4], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{AI4}/requirements.txt", "-r", f"{AI4}/requirements-dev.txt"])

env = dict(os.environ, PYTHONPATH=f"{AI4}:{os.getcwd()}")
subprocess.run([sys.executable, "-m", "extraction.demo", "--today", TODAY, "--out", "results/ai3_week3/demo.md"],
               env=env, check=True, capture_output=True)
display(Markdown(Path("results/ai3_week3/demo.md").read_text(encoding="utf-8")))

r = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_integration.py", "-rA"],
                   env=dict(os.environ, AI_TTS_WEB=AI4), capture_output=True, text=True)
print(r.stdout[-1500:])

## 7. 직접 넣어 보기

글자를 줄바꿈으로 나눠 넣으면 OCR 결과 대신 쓸 수 있습니다. 3주차에 더한 예외 처리 예시도 같이 넣었습니다.

In [ ]:
examples = {
    "기본": """제품명: 고소한 두유
원재료명: 정제수, 대두(국산), 설탕
이 제품은 땅콩, 호두를 사용한 제품과 같은 제조시설에서 제조하고 있습니다.
소비기한 26.11.20까지""",
    "기한이 서로 다르게 읽힘": "소비기한 2026.12.01\n소비기한 2026.12.07",
    "알레르기 목록이 잘림": "알레르기 유발물질: 밀, 우유, 대두,\n소비기한 2026.12.10까지",
    "수입품 영어 표시": "Contains: milk, soy. May contain peanuts.\nBEST BEFORE 2027.01.10",
}
for name, text in examples.items():
    print(f"■ {name}")
    print(summary(extract(text, ["땅콩"], TODAY)), end="\n\n")